# Reproducing GLOT Paper Results with Original Code

This notebook runs the **original GLOT implementation** from [ipsitmantri/GLOT](https://github.com/ipsitmantri/GLOT) to reproduce the paper's reported results.

**Experiments:**
1. GLUE benchmark (9 tasks × 5 poolers × 2 encoder backbones)
2. Diagnostic stress test (4 distractor ratios × 5 poolers × 2 backbones)

**Hardware:** Requires GPU. Go to `Runtime → Change runtime type → T4 GPU` (or A100 for decoder models).

**Time estimate:** ~2-4 hours for all encoder experiments on T4.

In [ ]:
import torch, os, json, subprocess, time
from google.colab import drive, userdata

# GPU check
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU not available. Go to Runtime → Change runtime type → T4 GPU"
    )
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")

# Mount Google Drive for persistent results
drive.mount("/content/drive")
RESULTS_DIR = "/content/drive/MyDrive/glot_reproduction"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {RESULTS_DIR}")

In [ ]:
import sys

ORG_REPO = "https://github.com/ipsitmantri/GLOT.git"
ORG_DIR = "/content/org_glot"

if not os.path.exists(ORG_DIR):
    !git clone {ORG_REPO} {ORG_DIR}
else:
    !cd {ORG_DIR} && git pull

# Show repo contents
!ls -la {ORG_DIR}

In [ ]:
# Detect Colab CUDA version and install compatible PyG
import subprocess
cuda_version = torch.version.cuda  # e.g. "12.4"
print(f"PyTorch CUDA: {cuda_version}")

# Install their requirements, but handle PyG wheels for Colab's CUDA
# Their requirements.txt pins torch-2.8.0+cu129 which may not match Colab
# We install PyG components compatible with the existing torch/CUDA

!pip install -q torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv \
    transformers datasets sentence-transformers mteb peft wandb accelerate \
    scikit-learn numpy pandas tqdm matplotlib seaborn polars

print("Dependencies installed")

In [ ]:
# HuggingFace token for gated models
try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    HF_TOKEN = ""
    print("WARNING: No HF_TOKEN found. Gated models (Llama, Mistral) will fail.")
    print("Add your token in Colab: Settings (gear icon) → Secrets → HF_TOKEN")

# Patch the HF_TOKEN in main.py so it doesn't use the placeholder
main_py = os.path.join(ORG_DIR, "main.py")
with open(main_py, "r") as f:
    content = f.read()
content = content.replace('HF_TOKEN = "<>"', f'HF_TOKEN = os.environ.get("HF_TOKEN", "")')
with open(main_py, "w") as f:
    f.write(content)

# Set wandb to offline mode (no account needed)
os.environ["WANDB_MODE"] = "offline"
print("W&B set to offline mode (results captured from stdout)")